## Imports

In [8]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import os
import sys
import matplotlib.pyplot as plt

os.chdir("/home/patrick/ansermodelling")

from models.FFNN_network import FFNN
from data.anser_dataset import *
from models.train import *
from models.model_wrappers import NNSolver
from models.eval import report_error_stats

## Dataset

In [2]:
train_loader, test_loader = make_dataloaders("data/dataset.npz")

In [3]:
model = FFNN(input_dim=8, output_dim=6, hidden_dims=[256,256,256])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = PoseLoss()

In [4]:
history = train(model, train_loader, test_loader, optimizer, epochs=200, loss_fn = loss_fn, print_losses = True)

epoch 	 LR 	 Training Loss 	 Test Loss 	 e_p test 	 e_n test
1 	 1.00e-03 	 0.0107 	  0.0061 	 37.6061 	 20.6354
2 	 1.00e-03 	 0.0046 	  0.0044 	 33.5704 	 17.1702
3 	 1.00e-03 	 0.0036 	  0.0035 	 27.7631 	 15.5802
4 	 1.00e-03 	 0.0031 	  0.0033 	 25.7584 	 14.7044
5 	 1.00e-03 	 0.0027 	  0.0026 	 23.6460 	 12.1103
6 	 1.00e-03 	 0.0025 	  0.0025 	 22.4954 	 12.1992
7 	 1.00e-03 	 0.0023 	  0.0024 	 23.9403 	 11.7325
8 	 1.00e-03 	 0.0022 	  0.0022 	 22.6719 	 10.6987
9 	 1.00e-03 	 0.0021 	  0.0021 	 21.7767 	 10.5689
10 	 1.00e-03 	 0.0020 	  0.0021 	 22.7004 	 10.4976
11 	 1.00e-03 	 0.0020 	  0.0021 	 21.8345 	 10.5958
12 	 1.00e-03 	 0.0019 	  0.0022 	 23.5715 	 10.8673
13 	 1.00e-03 	 0.0019 	  0.0020 	 21.4570 	 10.0974
14 	 1.00e-03 	 0.0018 	  0.0020 	 22.7864 	 10.1777
15 	 1.00e-03 	 0.0018 	  0.0019 	 21.4248 	 9.8335
16 	 1.00e-03 	 0.0018 	  0.0019 	 22.2358 	 9.6250
17 	 1.00e-03 	 0.0018 	  0.0019 	 20.7673 	 9.8885
18 	 1.00e-03 	 0.0018 	  0.0018 	 19.6565 	 9.545

In [5]:
torch.save(model.state_dict(), "models/checkpoints/nn_normal_poseloss_noscheduler.pt")

In [6]:
test_set = np.load("data/test_set.npz")
measurements = test_set["xs"]
poses = test_set["ys"]

In [9]:
nn = NNSolver(model)

In [11]:
%%time
poses_pred_nn, success_nn = nn.solve(measurements)

CPU times: user 410 ms, sys: 28.2 ms, total: 438 ms
Wall time: 92.7 ms


In [12]:
print("With normal and poseloss (no scheduler)")
report_error_stats(poses_pred_nn,poses,success_nn)

With normal and poseloss (no scheduler)
Mean pos error: 21.3, mean angle error: 8.67
Median pos error: 20.1, Median angle error: 5.92
95% pos error : 38.7 95% angle error: 25.4
LM success rate: 1
Convergence rate: 0
Mean pos error of converged: nan, mean angle error of converged: nan 


/home/patrick/ansermodelling/models/eval.py:30: RuntimeWarning: Mean of empty slice
  print(f"Mean pos error of converged: {ex_conv.mean():.3g}, mean angle error of converged: {en_conv.mean():.3g} ")
/home/patrick/ansermodelling/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
